<h3>형식에 맞는 문서 단위</h3>
<p><strong>개념</strong> DOCX는 문단과 표, HTML은 본문 영역, CSV는 행이 기본 단위입니다. 하나의 범용 문자열 변환보다 형식별 구조를 먼저 읽습니다.</p>

In [1]:
%pip install -q "beautifulsoup4==4.15.0" "python-docx==1.2.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from bs4 import BeautifulSoup
from docx import Document
import csv

roots = [Path.cwd(), *Path.cwd().parents]
labs = [p / "labs/part3/ch10-document-processing" for p in roots
        if (p / "labs/part3/ch10-document-processing").is_dir()]
if not labs: raise FileNotFoundError("저장소 루트에서 노트북을 실행하세요")
lab = labs[0]
fixtures = lab / "fixtures"


<h3>세 형식을 짧게 읽기</h3>
<p><strong>확인할 점</strong> 각 결과가 어떤 단위를 보존하는지 출력에서 확인합니다.</p>

In [3]:
docx = Document(fixtures / "sample.docx")
paragraphs = [p.text for p in docx.paragraphs if p.text.strip()]
soup = BeautifulSoup((fixtures / "sample.html").read_text(), "html.parser")
for tag in soup(["script", "style"]): tag.decompose()
with (fixtures / "sample.csv").open(encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))
print({"docx_paragraphs": len(paragraphs),
       "html": soup.get_text(" ", strip=True)[:50], "csv_rows": len(rows)})


{'docx_paragraphs': 2, 'html': '반품 안내 수령 후 7일 이내 신청합니다.', 'csv_rows': 2}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 존재하지 않는 CSV 열을 읽으면 스키마 불일치를 즉시 확인할 수 있습니다.</p>

In [4]:
try:
    print(rows[0]["없는_열"])
except KeyError as error:
    print(f"caught: CSV schema mismatch {error}")


caught: CSV schema mismatch '없는_열'
